In [3]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, average_precision_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as make_imb_pipeline   # SMOTE를 학습할 때만 적용하는 파이프라인
from xgboost import XGBClassifier

SEED = 42                      # 결과를 매번 같게
HOUR, DAY = 3600, 86400        # 시간 단위(초)

# 노트북을 프로젝트 루트나 models/ 어디서 열어도 데이터 폴더를 찾는다
cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, cwd.parent) if (path / 'data/processed/step_s1').exists())
PROC = ROOT / 'data/processed'

PRESSURE, FLOW = 'FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE'     # 냉각 압력, 냉각 유량
DROP_FAULT = 'FlowCool Pressure Dropped Below Limit'        # 압력 저하
DROP_TTF = 'TTF_' + DROP_FAULT                              # 압력 저하까지 남은 시간(초) 열

In [4]:
load_cols = ['Tool', 'time_start', 'time_end', 'recipe', 'recipe_step', DROP_TTF, PRESSURE + '_mean', FLOW + '_mean']
segments = pd.concat([pd.read_parquet(path, columns=load_cols) for path in sorted((PROC / 'step_s1').glob('*.parquet'))],
                     ignore_index=True)
segments = segments.sort_values(['Tool', 'time_start']).reset_index(drop=True)   # 장비별 시간순

faults = pd.read_csv(PROC / 'faults.csv')
faults['Tool'] = faults.Tool.str[2:] + '_' + faults.Tool.str[:2]                      # 01M01 → M01_01 (데이터와 이름 맞추기)
faults['fault_type'] = faults.fault_name.str.replace(' [NoWaferID]', '', regex=False)   # 같은 고장의 표기 차이 통일
faults = faults.drop_duplicates(['Tool', 'time', 'fault_type']).sort_values(['Tool', 'time']).reset_index(drop=True)

print(f'스텝 구간 {len(segments):,}행 · 장비 {segments.Tool.nunique()}대 · 고장 기록 {len(faults):,}건')
segments.head()

스텝 구간 526,268행 · 장비 20대 · 고장 기록 1,211건


,Tool,time_start,time_end,recipe,recipe_step,TTF_FlowCool Pressure Dropped Below Limit,FLOWCOOLPRESSURE_mean,FLOWCOOLFLOWRATE_mean
0,M01_01,3283542,3283842,3,3.0,3249342.0,0.424550,0.686933
1,M01_01,3284286,3284586,3,3.0,3248598.0,0.420978,0.687450
2,M01_01,3285074,3285970,2,3.0,3247214.0,0.428303,0.687351
3,M01_01,3287330,3287686,1,3.0,3245498.0,0.410633,0.686505
4,M01_01,3288170,3288310,2,3.0,3244874.0,0.429895,0.687712


In [5]:
tool_first = segments.groupby('Tool').time_start.min()     # 장비별 기록 시작
tool_last = segments.groupby('Tool').time_end.max()        # 장비별 기록 끝
test_start = tool_first + (tool_last - tool_first) * 0.75  # 평가 시작 시각

row_test_start = segments.Tool.map(test_start)
segments['part'] = np.select([segments.time_end < row_test_start - DAY,      # 평가 시작 하루 전까지 → 학습
                              segments.time_start >= row_test_start],        # 평가 시작 이후 → 평가
                             ['train', 'test'], default='gap')               # 사이 하루 → 사용 안 함
segments.part.value_counts()

part
train    398281
test     126754
gap        1233
Name: count, dtype: int64

In [6]:
ttf_drop = segments[DROP_TTF]
segments['label'] = (ttf_drop <= DAY).astype(int)

is_unknown = ((segments.Tool.map(tool_last) - segments.time_end) < DAY) & ttf_drop.isna()
segments = segments[~is_unknown].reset_index(drop=True)
print(f'판단 불가로 뺀 행: {is_unknown.sum():,}')
segments.groupby('part').label.agg(['size', 'sum', 'mean']).rename(
    columns={'size': '행 수', 'sum': '정답 1', 'mean': '정답 1 비율'}).round(4)

판단 불가로 뺀 행: 1,257


,행 수,정답 1,정답 1 비율
part,,,
gap,1233,9,0.0073
test,125497,4503,0.0359
train,398281,26828,0.0674
